# 03 Clustering and Eval

Here we do the actual data mining work, selecting clustering features, (prolly gonna be reclat, reclong, and the normalized masses and distance from equator which we create in the previous section as its own feature)

Clustering will be K-means, where we run the cluster for a given k in range, check the silhouette score, and determine the best K that we then use in the .py

In [5]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Alright, now that we have a processed dataset, we start doing clustering and eval of said clusters.
# This will be done on 3 subsets of the main dataset, full fell and found, with more features being
# subset and clustered later.

# loading the dataset
df = pd.read_csv("../data/processed/meteorite_landings_processed.csv")

# subsets (CAN ADD MORE LATER, FOR NOW WE JUST USE THESE 3)

subsets = {} # list of all of our subsets.
subsets_continents = {}

# main
subsets["main"] = df.copy()

# fell vs found
subsets["fell"] = df[df["fall_binary"] == 1].copy()
subsets["found"] = df[df["fall_binary"] == 0].copy()

# all valid geo locations
valid_geo = (
    df["reclat"].between(-90, 90)
    & df["reclong_norm"].between(-180, 180)
    & ~((df["reclat"] == 0) & (df["reclong_norm"] == 0))
)
subsets["geo_valid"] = df[valid_geo].copy()

# land only 
land_mask = valid_geo & (~df["continent"].isin(["Open Ocean", "Unknown"]))
subsets["land_only"] = df[land_mask].copy()

# modern and historic for comparisons of bias?
subsets["modern"] = df[df["year"] >= 1950].copy()
subsets["historic"] = df[df["year"] < 1950].copy()

for cont, n in df["continent"].value_counts().items(): # gets each continent and its total num records
    if cont in ["Open Ocean", "Unknown"]: # skips if non continent
        continue
    if n >= 500: # if continent has at least 500, it gets a subset!
        key = "cont_" + cont.lower().replace(" ", "_") # creates the key, cont_continentname
        subsets_continents[key] = df[df["continent"] == cont].copy() # adds it to subsets!


print("Primary subsets:")
for k, v in subsets.items():
    print(f"{k:20s} {v.shape}")

print("\nContinent subsets:")
for k, v in subsets_continents.items():
    print(f"{k:20s} {v.shape}")



Primary subsets:
main                 (45716, 16)
fell                 (1107, 16)
found                (44609, 16)
geo_valid            (32187, 16)
land_only            (32093, 16)
modern               (43740, 16)
historic             (1685, 16)

Continent subsets:
cont_antarctica      (22099, 16)
cont_asia            (3562, 16)
cont_africa          (2845, 16)
cont_north_america   (1822, 16)
cont_oceania         (648, 16)
cont_europe          (617, 16)
cont_south_america   (500, 16)


In [ ]:
# Now for the clustering! first, we want to identify our feature sets that we will use for clustering.
feature_sets = {
    "baseline" : ["log_mass", "reclat", "reclong_norm", "year"],
    "time_space" : ["year", "reclat", "reclong_norm"],
    "mass_space" : ["log_mass", "reclat", "reclong_norm"],
    "full" : ["log_mass", "reclat", "reclong_norm", "year", "fall_binary"]
}

# next we define a function running the kmeans clustering and eval for a given subset and feature set
def run_clustering_and_eval(subset, features, k_range=range(2, 11), n_init=20):
    
    # first we want only the selected features, and non empty rows or values.
    X = subset[features].apply(pd.to_numeric, errors='coerce').dropna() # converts to numeric and drops non numeric rows    

    if len(X) < 3:
        raise ValueError("Not enough rows after dropping NaN's")
    
    # scale for k-means
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    rows = [] # each row
    models = {} # the k_means we have made so far, will get best at the end

    # valid K range
    valid_ks = [k for k in k_range if 2 <= k <= (len(X) - 1)]

    for k in valid_ks:
        kmeans = KMeans(n_clusters=k, n_init=n_init) # initializes the kmeans model
        labels = kmeans.fit_predict(X_scaled) # labels for each point
        
        if len(np.unique(labels)) < 2: # if only one cluster, silhouette score is not defined, skip
            continue
        
        rows.append({
            "k": k, # number of clusters
            "silhouette": silhouette_score(
                X_scaled,
                labels,
                sample_size=min(5000, len(X_scaled)),
                random_state=42
            ),            
            "inertia": kmeans.inertia_ # cluster tightness
            })
        
        models[k] = kmeans # saves the clustering model

    # scores dataframe, sorted by silhouette score, best first
    scores = pd.DataFrame(rows).sort_values("silhouette", ascending=False)

    if scores.empty:
        raise ValueError("No valid silhouette results for this subset/features.")

    best_k = int(scores.iloc[0]["k"]) # gets our best performing K!
    best_model = models[best_k]

    clustered = subset.loc[X.index].copy()
    clustered["cluster"] = best_model.labels_


    return {
        "best_k": best_k,
        "scores": scores,           # all tested k results
        "model": best_model,        # fitted KMeans for best k
        "scaler": scaler,           # fitted scaler for this subset
        "clustered_df": clustered,  # rows used + cluster labels
        "feature_cols": features
    }


# next we call the clustering for each of our subsets and features sets.
res_main_with_fall = run_clustering_and_eval(subsets["main"], feature_sets["full"]) # main with full featureset

res_main = run_clustering_and_eval(subsets["main"], feature_sets["baseline"]) # main with baseline
res_fell = run_clustering_and_eval(subsets["fell"], feature_sets["baseline"]) # fell with baseline
res_found = run_clustering_and_eval(subsets["found"], feature_sets["baseline"]) # found with baseline

res_geo_valid = run_clustering_and_eval(subsets["geo_valid"], feature_sets["baseline"]) # only valid geo with baseline
res_land_only = run_clustering_and_eval(subsets["land_only"], feature_sets["baseline"]) # only land with baseline

res_modern = run_clustering_and_eval(subsets["modern"], feature_sets["baseline"]) # only modern with baseline
res_historic = run_clustering_and_eval(subsets["historic"], feature_sets["baseline"]) # only historic with baseline


In [9]:
# our results!
run_results = {
    "main_baseline": res_main,
    "main_with_fall": res_main_with_fall,
    "fell_baseline": res_fell,
    "found_baseline": res_found,
    "geo_valid_baseline": res_geo_valid,
    "land_only_baseline": res_land_only,
    "modern_baseline": res_modern,
    "historic_baseline": res_historic,
}

# summary of results sorted by score
summary_rows = []
for name, r in run_results.items():
    best = r["scores"].iloc[0]
    summary_rows.append({
        "run": name,
        "n_rows_clustered": len(r["clustered_df"]),
        "best_k": int(r["best_k"]),
        "best_silhouette": float(best["silhouette"]),
        "best_inertia": float(best["inertia"]),
        "features": ", ".join(r["feature_cols"]),
    })

summary = pd.DataFrame(summary_rows).sort_values("best_silhouette", ascending=False)
display(summary)

# cluster centers for each run transformed back to original feature space
centers = {}
for name, r in run_results.items():
    c = r["scaler"].inverse_transform(r["model"].cluster_centers_)
    centers[name] = pd.DataFrame(c, columns=r["feature_cols"])
    centers[name]["cluster"] = centers[name].index

# and save to file so we dont have to do this again!
summary.to_csv("../outputs/tables/kmeans_summary.csv", index=False)

for name, r in run_results.items():
    r["clustered_df"].to_csv(f"../outputs/tables/{name}_clustered.csv", index=False)
    centers[name].to_csv(f"../outputs/tables/{name}_centers.csv", index=False)

,run,n_rows_clustered,best_k,best_silhouette,best_inertia,features
4,geo_valid_baseline,31929,3,0.483193,55158.482242,"log_mass, reclat, reclong_norm, year"
5,land_only_baseline,31839,3,0.482108,55050.226133,"log_mass, reclat, reclong_norm, year"
1,main_with_fall,38115,4,0.462583,67153.559399,"log_mass, reclat, reclong_norm, year, fall_binary"
0,main_baseline,38115,3,0.458727,66129.436934,"log_mass, reclat, reclong_norm, year"
3,found_baseline,37050,3,0.448551,65474.339611,"log_mass, reclat, reclong_norm, year"
6,modern_baseline,36479,2,0.424864,84680.868137,"log_mass, reclat, reclong_norm, year"
7,historic_baseline,1636,4,0.328015,3253.658564,"log_mass, reclat, reclong_norm, year"
2,fell_baseline,1065,9,0.254347,1428.328931,"log_mass, reclat, reclong_norm, year"
